# 🧠 Deep Learning untuk Analisis Sentimen Review Film (IMDB)

**Workshop Pemrosesan Bahasa Alami (PBA)**  
*Institut Teknologi Sumatera (ITERA)*  

---

## 🎯 Tujuan

Pada proyek ini, kita akan membangun dan membandingkan **dua model deep learning** untuk mengklasifikasikan sentimen review film menjadi:

- **Positive**
- **Negative**

---

## 🤖 Model yang Digunakan

| # | Model | Konsep Utama |
|--|------|-------------|
| 1 | **BiLSTM** | Recurrent neural network dua arah |
| 2 | **BiLSTM + Attention** | Fokus pada kata penting dalam teks |

---

## 📊 Dataset

- Dataset: **IMDB Movie Reviews (50K)**
- Sumber: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews  
- Digunakan: **Subset (±3.000 – 10.000 data)**

---

## 🧾 Label

- **positive → 1**
- **negative → 0**

---

## ⚙️ Framework

- PyTorch
- Scikit-learn
- Pandas

---

## 🏗️ Arsitektur Model

### 🔹 BiLSTM

## 📘 1. Import Library
Penjelasan

Mengimpor semua library dan module yang dibutuhkan untuk preprocessing, training, dan evaluasi model.

In [1]:
import pandas as pd
import torch

from config import *
from preprocess import preprocess
from dataset import Vocabulary, get_lstm_dataloaders
from models import BiLSTMClassifier, BiLSTMAttentionClassifier
from train import set_seed, train_model, evaluate

## 📘 2. Set Seed & Device
📌 Penjelasan

Digunakan untuk memastikan hasil eksperimen konsisten dan menentukan apakah menggunakan CPU atau GPU.

In [2]:
set_seed(RANDOM_SEED)
print("Device:", DEVICE)

Device: cpu


## 📘 3. Load & Preprocess Dataset
📌 Penjelasan

Dataset IMDB (50K review) dimuat lalu diambil subset (±10K/3K sesuai config). Teks dibersihkan (cleaning) dan label di-encode menjadi angka.

In [3]:
df = preprocess()

print("Jumlah data:", len(df))
df.head()

📥 Load dataset...
PATH RAW: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\Data\imdb-dataset-of-50k-movie-reviews\IMDB Dataset.csv
EXIST: True
📊 Jumlah data awal: 50,000
🔀 Setelah sampling: 10,000
🧹 Cleaning text...
🔢 Encoding label...
📊 Setelah cleaning: 10,000
✅ Data bersih disimpan di: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\Data\clean_imdb_10k.csv
Jumlah data: 10000


,review,sentiment,clean_review,label_encoded
0,This was obviously the worst movie ever made.....,negative,this was obviously the worst movie ever made k...,0
1,Fame did something odd. It was not only a musi...,negative,fame did something odd it was not only a music...,0
2,"From all the rave reviews, we couldn't wait to...",negative,from all the rave reviews we couldn t wait to ...,0
3,"First, let me confess that I have not read thi...",positive,first let me confess that i have not read this...,1
4,Mickey Rourke is enjoying a renaissance at the...,negative,mickey rourke is enjoying a renaissance at the...,0


## 📘 4. Build Vocabulary
📌 Penjelasan

Mengubah kata menjadi angka (tokenisasi sederhana) agar bisa diproses oleh model deep learning.

In [4]:
vocab = Vocabulary()
vocab.build_vocab(df["clean_review"].tolist(), max_size=VOCAB_SIZE)

print("Ukuran vocab:", len(vocab.word2idx))

📖 Vocabulary: 5,000 kata
Ukuran vocab: 5000


## 📘 5. Split Data & DataLoader
📌 Penjelasan

Dataset dibagi menjadi:

Train
Validation
Test

Kemudian dibuat DataLoader untuk proses training.

In [5]:
train_loader, val_loader, test_loader = get_lstm_dataloaders(df, vocab)

📦 Train: 6999 | Val: 1000 | Test: 2001


## 📘 6. Training Model 1 — BiLSTM
📌 Penjelasan

Model BiLSTM digunakan untuk menangkap konteks kata dari dua arah (forward & backward).

In [6]:
model_lstm = BiLSTMClassifier(
    vocab_size=len(vocab)
)
history_lstm = train_model(
    model_lstm,
    train_loader,
    val_loader,
    test_loader,
    save_path=BILSTM_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE,
    model_name="bilstm"   # 
)


🚀 Training bilstm...

Epoch 1
Train Loss: 0.6934 | Acc: 0.5188
Val   Loss: 0.6871 | Acc: 0.5480
Epoch 2
Train Loss: 0.6796 | Acc: 0.5679
Val   Loss: 0.6754 | Acc: 0.5630

📊 Evaluasi bilstm...

📄 Classification Report:
              precision    recall  f1-score   support

    negative       0.55      0.63      0.59      1001
    positive       0.57      0.49      0.52      1000

    accuracy                           0.56      2001
   macro avg       0.56      0.56      0.55      2001
weighted avg       0.56      0.56      0.55      2001

Accuracy: 0.5567216391804098
F1 Score: 0.5546185749090374
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\confusion_matrix_bilstm.png
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\training_curves_bilstm.png


## 📘 7. Training Model 2 — BiLSTM + Attention
📌 Penjelasan

Menambahkan mekanisme attention agar model fokus pada kata penting dalam kalimat.

In [8]:
model_att = BiLSTMAttentionClassifier(
    vocab_size=len(vocab)
)
history_att = train_model(
    model_att,
    train_loader,
    val_loader,
    test_loader,
    save_path=BILSTM_ATT_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE,
    model_name="bilstm_attention"
)


🚀 Training bilstm_attention...

Epoch 1
Train Loss: 0.6855 | Acc: 0.5635
Val   Loss: 0.6644 | Acc: 0.6280
Epoch 2
Train Loss: 0.6388 | Acc: 0.6354
Val   Loss: 0.6788 | Acc: 0.6390

📊 Evaluasi bilstm_attention...

📄 Classification Report:
              precision    recall  f1-score   support

    negative       0.74      0.34      0.47      1001
    positive       0.57      0.88      0.69      1000

    accuracy                           0.61      2001
   macro avg       0.66      0.61      0.58      2001
weighted avg       0.66      0.61      0.58      2001

Accuracy: 0.6101949025487257
F1 Score: 0.5796422479863831
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\confusion_matrix_bilstm_attention.png
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\training_curves_bilstm_attention.png


In [9]:
# ======================
# MODEL
# ======================
model = BiLSTMClassifier(
    vocab_size=len(vocab.word2idx),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM
)

# ======================
# TRAINING (INI YANG KAMU TANYA)
# ======================
history = train_model(
    model,
    train_loader,
    val_loader,
    test_loader,
    save_path=BILSTM_MODEL_PATH,
    epochs=LSTM_EPOCHS,
    lr=LSTM_LR,
    patience=LSTM_PATIENCE
)



🚀 Training model...

Epoch 1
Train Loss: 0.6942 | Acc: 0.5134
Val   Loss: 0.6905 | Acc: 0.5220
Epoch 2
Train Loss: 0.6832 | Acc: 0.5602
Val   Loss: 0.6855 | Acc: 0.5470

📊 Evaluasi model...

📄 Classification Report:
              precision    recall  f1-score   support

    negative       0.57      0.43      0.49      1001
    positive       0.54      0.68      0.60      1000

    accuracy                           0.56      2001
   macro avg       0.56      0.56      0.55      2001
weighted avg       0.56      0.56      0.55      2001

Accuracy: 0.5552223888055972
F1 Score: 0.5485374521769796
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\confusion_matrix_model.png
📊 Saved: c:\Users\USER\Documents\Tubes NLP\pba2026-kelompok-3\DL\plots\training_curves_model.png


## 📘 9. Kesimpulan
📌 Penjelasan
BiLSTM mampu memahami urutan kata dalam kalimat
BiLSTM + Attention biasanya memberikan performa lebih baik karena fokus pada kata penting
Dataset yang digunakan adalah IMDB Review dengan klasifikasi sentimen (positif/negatif)